In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected.")
    print("In Colab: Runtime → Change runtime type → T4 GPU")

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4
CUDA version: 12.8


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("SALT-NLP/ImplicitHate")

print(dataset)

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


implicit_hate.csv:   0%|          | 0.00/708k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6346 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['post', 'implicit_class', 'extra_implicit_class'],
        num_rows: 6346
    })
})


In [ ]:
print("Available splits:")

for split in dataset:
    print(f"{split}: {len(dataset[split])} examples")

Available splits:
train: 6346 examples


In [ ]:
for split in dataset:
    print(f"\n{split} columns:")
    print(dataset[split].column_names)

example = dataset[list(dataset.keys())[0]][0]

example


train columns:
['post', 'implicit_class', 'extra_implicit_class']


{'post': '  " : jewish harvard professor noel ignatiev wants to abolish the white race via #wr " " "',
 'implicit_class': 'white_grievance',
 'extra_implicit_class': None}

In [ ]:
from collections import Counter

labels = Counter(dataset["train"]["implicit_class"])

for label, count in labels.most_common():
    print(f"{label}: {count}")

white_grievance: 1538
incitement: 1269
stereotypical: 1133
inferiority: 863
irony: 797
threatening: 666
other: 80


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = dataset["train"].to_pandas()

# Remove "other"
df = df[df["implicit_class"] != "other"].copy()


df["category"] = df["implicit_class"].str.rsplit("_", n=1).str[-1]

# Rename categories
category_names = {
    "grievance": "Grievance",
    "incitement": "Incitement",
    "stereotypical": "Stereotypes",
    "inferiority": "Inferiority",
    "irony": "Irony",
    "threatening": "Threats"
}

df["category"] = df["category"].map(category_names)


print("Category counts:")
print(df["category"].value_counts())

Category counts:
category
Grievance      1538
Incitement     1269
Stereotypes    1133
Inferiority     863
Irony           797
Threats         666
Name: count, dtype: int64


In [ ]:
# 80% train, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["category"],
    random_state=42
)


# 10% development, 10% test
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["category"],
    random_state=42
)

print("Train:", len(train_df))
print("Dev:", len(dev_df))
print("Test:", len(test_df))

Train: 5012
Dev: 627
Test: 627


In [ ]:
print("TRAIN")
print(train_df["category"].value_counts())
print()

print("DEV")
print(dev_df["category"].value_counts())
print()

print("TEST")
print(test_df["category"].value_counts())

TRAIN
category
Grievance      1230
Incitement     1015
Stereotypes     906
Inferiority     690
Irony           638
Threats         533
Name: count, dtype: int64

DEV
category
Grievance      154
Incitement     127
Stereotypes    113
Inferiority     86
Irony           80
Threats         67
Name: count, dtype: int64

TEST
category
Grievance      154
Incitement     127
Stereotypes    114
Inferiority     87
Irony           79
Threats         66
Name: count, dtype: int64


In [ ]:
print("TRAIN (%)")
print((train_df["category"].value_counts(normalize=True) * 100).round(2))
print()

print("DEV (%)")
print((dev_df["category"].value_counts(normalize=True) * 100).round(2))
print()

print("TEST (%)")
print((test_df["category"].value_counts(normalize=True) * 100).round(2))

TRAIN (%)
category
Grievance      24.54
Incitement     20.25
Stereotypes    18.08
Inferiority    13.77
Irony          12.73
Threats        10.63
Name: proportion, dtype: float64

DEV (%)
category
Grievance      24.56
Incitement     20.26
Stereotypes    18.02
Inferiority    13.72
Irony          12.76
Threats        10.69
Name: proportion, dtype: float64

TEST (%)
category
Grievance      24.56
Incitement     20.26
Stereotypes    18.18
Inferiority    13.88
Irony          12.60
Threats        10.53
Name: proportion, dtype: float64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


import os

save_path = "/content/drive/MyDrive/implicit_hate_project"
os.makedirs(save_path, exist_ok=True)


train_df.to_csv(f"{save_path}/train.csv", index=False)
dev_df.to_csv(f"{save_path}/dev.csv", index=False)
test_df.to_csv(f"{save_path}/test.csv", index=False)

print("Saved successfully!")
print(f"Files are in: {save_path}")

Mounted at /content/drive
Saved successfully!
Files are in: /content/drive/MyDrive/implicit_hate_project


In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd

save_path = "/content/drive/MyDrive/implicit_hate_project"

train_df = pd.read_csv(f"{save_path}/train.csv")
dev_df = pd.read_csv(f"{save_path}/dev.csv")
test_df = pd.read_csv(f"{save_path}/test.csv")

print("Train:", len(train_df))
print("Dev:", len(dev_df))
print("Test:", len(test_df))

Train: 5012
Dev: 627
Test: 627


In [ ]:
print(train_df.columns)
print(train_df.head())

Index(['post', 'implicit_class', 'extra_implicit_class', 'category'], dtype='object')
                                                post implicit_class  \
0  multiple murders by fireblack people kill 85 y...  stereotypical   
1  have to hire illegals in california to compete...    inferiority   
2  this is an inherently white thing to do . keep...     incitement   
3  the immigration act of 1790 stipulated white b...     incitement   
4  i'm a white american and i want trump to be ki...     incitement   

  extra_implicit_class     category  
0                  NaN  Stereotypes  
1                  NaN  Inferiority  
2                  NaN   Incitement  
3                  NaN   Incitement  
4                  NaN   Incitement  


In [ ]:
label2id = {
    "Grievance": 0,
    "Incitement": 1,
    "Stereotypes": 2,
    "Inferiority": 3,
    "Irony": 4,
    "Threats": 5
}

id2label = {v: k for k, v in label2id.items()}

train_df["label"] = train_df["category"].map(label2id)
dev_df["label"] = dev_df["category"].map(label2id)
test_df["label"] = test_df["category"].map(label2id)

print(train_df[["category", "label"]].head())

      category  label
0  Stereotypes      2
1  Inferiority      3
2   Incitement      1
3   Incitement      1
4   Incitement      1


In [ ]:
print("Missing train text:", train_df["post"].isna().sum())
print("Missing dev text:", dev_df["post"].isna().sum())
print("Missing test text:", test_df["post"].isna().sum())
print("Missing train labels:", train_df["label"].isna().sum())
print("Missing dev labels:", dev_df["label"].isna().sum())
print("Missing test labels:", test_df["label"].isna().sum())
print("Missing train labels:", train_df["label"].isna().sum())
print("Missing dev labels:", dev_df["label"].isna().sum())
print("Missing test labels:", test_df["label"].isna().sum())
print(sorted(train_df["label"].unique()))
print(sorted(dev_df["label"].unique()))
print(sorted(test_df["label"].unique()))
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Missing train text: 0
Missing dev text: 0
Missing test text: 0
Missing train labels: 0
Missing dev labels: 0
Missing test labels: 0
Missing train labels: 0
Missing dev labels: 0
Missing test labels: 0
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
GPU available: True
GPU: Tesla T4


In [ ]:
from transformers import AutoTokenizer

model_name = "roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [ ]:
def tokenize_data(df):
    return tokenizer(
        df["post"].tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_data(train_df)
dev_encodings = tokenize_data(dev_df)
test_encodings = tokenize_data(test_df)

print("Train examples:", len(train_encodings["input_ids"]))
print("Dev examples:", len(dev_encodings["input_ids"]))
print("Test examples:", len(test_encodings["input_ids"]))

Train examples: 5012
Dev examples: 627
Test examples: 627


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=6,
    id2label=id2label,
    label2id=label2id
)

print("RoBERTa classification model loaded!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa classification model loaded!


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from datasets import Dataset



model_name = "roberta-base"



tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_data(df):
    return tokenizer(
        df["post"].tolist(),
        padding=True,
        truncation=True,
        max_length=128
    )

train_encodings = tokenize_data(train_df)
dev_encodings = tokenize_data(dev_df)
test_encodings = tokenize_data(test_df)



train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_df["label"].tolist()
})

dev_dataset = Dataset.from_dict({
    "input_ids": dev_encodings["input_ids"],
    "attention_mask": dev_encodings["attention_mask"],
    "labels": dev_df["label"].tolist()
})

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_df["label"].tolist()
})



model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    id2label=id2label,
    label2id=label2id
)

print("RoBERTa model ready for fine-tuning.")
print("Train:", len(train_dataset))
print("Dev:", len(dev_dataset))
print("Test:", len(test_dataset))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RoBERTa model ready for fine-tuning.
Train: 5012
Dev: 627
Test: 627


In [ ]:
import torch
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score


if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Please enable the T4 GPU in Colab.")

device = torch.device("cuda")
print("Using GPU:", torch.cuda.get_device_name(0))


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted")
    }


training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/implicit_hate_project/roberta_baseline",


    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,


    eval_strategy="epoch",
    save_strategy="epoch",


    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,


    learning_rate=2e-5,
    weight_decay=0.01,


    logging_strategy="steps",
    logging_steps=50,


    fp16=True,


    seed=42,


    report_to="none"
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)


trainer.train()

Using GPU: Tesla T4


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.044904,0.964226,0.661882,0.657884,0.660859
2,0.837318,0.935379,0.682616,0.682933,0.680984
3,0.632707,0.932157,0.685805,0.685759,0.684499


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=942, training_loss=0.9472494753049944, metrics={'train_runtime': 142.1637, 'train_samples_per_second': 105.765, 'train_steps_per_second': 6.626, 'total_flos': 989069977663488.0, 'train_loss': 0.9472494753049944, 'epoch': 3.0})

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import numpy as np


test_results = trainer.predict(test_dataset)


test_predictions = np.argmax(test_results.predictions, axis=1)
test_labels = test_results.label_ids


accuracy = accuracy_score(test_labels, test_predictions)
macro_f1 = f1_score(test_labels, test_predictions, average="macro")
weighted_f1 = f1_score(test_labels, test_predictions, average="weighted")

print("TEST SET RESULTS")
print("----------------")
print(f"Accuracy:     {accuracy:.4f}")
print(f"Macro-F1:     {macro_f1:.4f}")
print(f"Weighted-F1:  {weighted_f1:.4f}")


print("\nPER-CATEGORY RESULTS")
print("--------------------")

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=[
            "Grievance",
            "Incitement",
            "Stereotypes",
            "Inferiority",
            "Irony",
            "Threats"
        ],
        digits=4
    )
)


cm = confusion_matrix(test_labels, test_predictions)

print("\nCONFUSION MATRIX")
print("----------------")
print(cm)

TEST SET RESULTS
----------------
Accuracy:     0.6475
Macro-F1:     0.6479
Weighted-F1:  0.6473

PER-CATEGORY RESULTS
--------------------
              precision    recall  f1-score   support

   Grievance     0.6824    0.6558    0.6689       154
  Incitement     0.5954    0.6142    0.6047       127
 Stereotypes     0.6726    0.6667    0.6696       114
 Inferiority     0.6310    0.6092    0.6199        87
       Irony     0.6857    0.6076    0.6443        79
     Threats     0.6173    0.7576    0.6803        66

    accuracy                         0.6475       627
   macro avg     0.6474    0.6518    0.6479       627
weighted avg     0.6494    0.6475    0.6473       627


CONFUSION MATRIX
----------------
[[101  22  14   8   1   8]
 [ 17  78  15   3   2  12]
 [ 15  11  76   6   3   3]
 [  5   9   3  53  14   3]
 [  6   4   4  12  48   5]
 [  4   7   1   2   2  50]]


In [ ]:
import os
import torch
import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer

project_dir = "/content/drive/MyDrive/implicit_hate_project"

label2id = {
    "Grievance": 0,
    "Incitement": 1,
    "Stereotypes": 2,
    "Inferiority": 3,
    "Irony": 4,
    "Threats": 5
}

id2label = {v: k for k, v in label2id.items()}


test_df = pd.read_csv(
    os.path.join(project_dir, "test.csv")
)

test_df["label"] = test_df["category"].map(label2id)


checkpoints = [
    "checkpoint-314",
    "checkpoint-628",
    "checkpoint-942"
]

for checkpoint in checkpoints:

    print("\n" + "=" * 60)
    print(checkpoint)
    print("=" * 60)

    model_path = os.path.join(
        project_dir,
        "roberta_baseline",
        checkpoint
    )


    model = AutoModelForSequenceClassification.from_pretrained(
        model_path
    )

    model = model.to("cuda")
    model.eval()

    print("Number of labels:", model.config.num_labels)
    print("Labels:", model.config.id2label)


    bias = model.classifier.out_proj.bias.detach().cpu().numpy()

    print("\nClassifier bias:")
    for i, value in enumerate(bias):
        print(f"{i} {id2label[i]:15s}: {value:.4f}")


    tokenizer = AutoTokenizer.from_pretrained(
        model_path
    )


    small_df = test_df.iloc[:10].copy()

    encodings = tokenizer(
        small_df["post"].tolist(),
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    encodings = {
        key: value.to("cuda")
        for key, value in encodings.items()
    }


    with torch.no_grad():
        logits = model(**encodings).logits

    predictions = torch.argmax(
        logits,
        dim=1
    ).cpu().numpy()

    print("\nFirst 10 predictions:")

    for i, pred in enumerate(predictions):
        print(
            f"{i}: "
            f"TRUE={small_df.iloc[i]['category']:15s} "
            f"PRED={id2label[int(pred)]}"
        )


    print("\nAverage logits for each class:")

    avg_logits = logits.mean(dim=0).cpu().numpy()

    for i, value in enumerate(avg_logits):
        print(
            f"{id2label[i]:15s}: {value:.4f}"
        )


checkpoint-314


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Number of labels: 6
Labels: {0: 'Grievance', 1: 'Incitement', 2: 'Stereotypes', 3: 'Inferiority', 4: 'Irony', 5: 'Threats'}

Classifier bias:
0 Grievance      : 0.0000
1 Incitement     : -0.0000
2 Stereotypes    : -0.0001
3 Inferiority    : 0.0000
4 Irony          : 0.0000
5 Threats        : 0.0000

First 10 predictions:
0: TRUE=Grievance       PRED=Inferiority
1: TRUE=Inferiority     PRED=Inferiority
2: TRUE=Grievance       PRED=Inferiority
3: TRUE=Irony           PRED=Inferiority
4: TRUE=Incitement      PRED=Inferiority
5: TRUE=Inferiority     PRED=Inferiority
6: TRUE=Inferiority     PRED=Inferiority
7: TRUE=Stereotypes     PRED=Inferiority
8: TRUE=Incitement      PRED=Inferiority
9: TRUE=Incitement      PRED=Inferiority

Average logits for each class:
Grievance      : -1.3256
Incitement     : -0.2481
Stereotypes    : -1.0509
Inferiority    : 1.6678
Irony          : 1.0781
Threats        : 0.1186

checkpoint-628


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Number of labels: 6
Labels: {0: 'Grievance', 1: 'Incitement', 2: 'Stereotypes', 3: 'Inferiority', 4: 'Irony', 5: 'Threats'}

Classifier bias:
0 Grievance      : 0.0000
1 Incitement     : 0.0000
2 Stereotypes    : -0.0001
3 Inferiority    : 0.0000
4 Irony          : 0.0000
5 Threats        : 0.0000

First 10 predictions:
0: TRUE=Grievance       PRED=Inferiority
1: TRUE=Inferiority     PRED=Inferiority
2: TRUE=Grievance       PRED=Inferiority
3: TRUE=Irony           PRED=Inferiority
4: TRUE=Incitement      PRED=Inferiority
5: TRUE=Inferiority     PRED=Inferiority
6: TRUE=Inferiority     PRED=Inferiority
7: TRUE=Stereotypes     PRED=Inferiority
8: TRUE=Incitement      PRED=Inferiority
9: TRUE=Incitement      PRED=Inferiority

Average logits for each class:
Grievance      : -1.6589
Incitement     : 1.0722
Stereotypes    : -1.5693
Inferiority    : 1.4404
Irony          : 0.8822
Threats        : -0.0577

checkpoint-942


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Number of labels: 6
Labels: {0: 'Grievance', 1: 'Incitement', 2: 'Stereotypes', 3: 'Inferiority', 4: 'Irony', 5: 'Threats'}

Classifier bias:
0 Grievance      : 0.0000
1 Incitement     : 0.0000
2 Stereotypes    : -0.0001
3 Inferiority    : 0.0000
4 Irony          : 0.0000
5 Threats        : -0.0000

First 10 predictions:
0: TRUE=Grievance       PRED=Inferiority
1: TRUE=Inferiority     PRED=Inferiority
2: TRUE=Grievance       PRED=Inferiority
3: TRUE=Irony           PRED=Inferiority
4: TRUE=Incitement      PRED=Inferiority
5: TRUE=Inferiority     PRED=Inferiority
6: TRUE=Inferiority     PRED=Inferiority
7: TRUE=Stereotypes     PRED=Inferiority
8: TRUE=Incitement      PRED=Inferiority
9: TRUE=Incitement      PRED=Inferiority

Average logits for each class:
Grievance      : -1.3051
Incitement     : 0.9623
Stereotypes    : -1.6254
Inferiority    : 1.2060
Irony          : 0.4192
Threats        : 0.3818


In [ ]:
import json
import os

state_path = "/content/drive/MyDrive/implicit_hate_project/roberta_baseline/checkpoint-942/trainer_state.json"

with open(state_path, "r") as f:
    state = json.load(f)

print("Best checkpoint:")
print(state.get("best_model_checkpoint"))

print("\nBest metric:")
print(state.get("best_metric"))

print("\nEvaluation history:")
for x in state["log_history"]:
    if "eval_macro_f1" in x:
        print(x)

Best checkpoint:
/content/drive/MyDrive/implicit_hate_project/roberta_baseline/checkpoint-942

Best metric:
0.6857588567188575

Evaluation history:
{'epoch': 1.0, 'eval_accuracy': 0.6618819776714514, 'eval_loss': 0.9642261862754822, 'eval_macro_f1': 0.6578835942538405, 'eval_runtime': 1.168, 'eval_samples_per_second': 536.812, 'eval_steps_per_second': 34.246, 'eval_weighted_f1': 0.6608594666029115, 'step': 314}
{'epoch': 2.0, 'eval_accuracy': 0.682615629984051, 'eval_loss': 0.9353794455528259, 'eval_macro_f1': 0.6829333896091062, 'eval_runtime': 1.2406, 'eval_samples_per_second': 505.417, 'eval_steps_per_second': 32.244, 'eval_weighted_f1': 0.6809838868020769, 'step': 628}
{'epoch': 3.0, 'eval_accuracy': 0.6858054226475279, 'eval_loss': 0.9321566224098206, 'eval_macro_f1': 0.6857588567188575, 'eval_runtime': 1.2657, 'eval_samples_per_second': 495.382, 'eval_steps_per_second': 31.603, 'eval_weighted_f1': 0.6844988354308569, 'step': 942}


In [ ]:
import pandas as pd
import os


project_dir = "/content/drive/MyDrive/implicit_hate_project"

prediction_path = os.path.join(
    project_dir,
    "test_predictions.csv"
)

df = pd.read_csv(prediction_path)


errors = df[df["true_label"] != df["predicted_label"]].copy()

print("Total test examples:", len(df))
print("Misclassified examples:", len(errors))
print("Correct examples:", len(df) - len(errors))


error_counts = (
    errors
    .groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values(
        ["true_label", "count"],
        ascending=[True, False]
    )
)

print("\n========================================")
print("ERROR COUNTS: TRUE → PREDICTED")
print("========================================")

print(error_counts.to_string(index=False))


print("\n========================================")
print("MISCLASSIFIED EXAMPLES")
print("========================================")

for true_label in errors["true_label"].unique():

    true_errors = errors[
        errors["true_label"] == true_label
    ]

    print("\n" + "=" * 70)
    print(f"TRUE CATEGORY: {true_label}")
    print("=" * 70)

    for predicted_label in true_errors["predicted_label"].unique():

        group = true_errors[
            true_errors["predicted_label"] == predicted_label
        ]

        print(
            f"\n--- Predicted as {predicted_label} "
            f"({len(group)} examples) ---"
        )

        for i, (_, row) in enumerate(group.iterrows(), start=1):

            print(f"\nExample {i}")
            print("Tweet:", row["post"])



error_output_path = os.path.join(
    project_dir,
    "misclassified_examples.csv"
)

errors.to_csv(
    error_output_path,
    index=False
)

print("\n========================================")
print("Saved misclassified examples to:")
print(error_output_path)
print("========================================")

Total test examples: 627
Misclassified examples: 221
Correct examples: 406

ERROR COUNTS: TRUE → PREDICTED
 true_label predicted_label  count
  Grievance      Incitement     22
  Grievance     Stereotypes     14
  Grievance     Inferiority      8
  Grievance         Threats      8
  Grievance           Irony      1
 Incitement       Grievance     17
 Incitement     Stereotypes     15
 Incitement         Threats     12
 Incitement     Inferiority      3
 Incitement           Irony      2
Inferiority           Irony     14
Inferiority      Incitement      9
Inferiority       Grievance      5
Inferiority     Stereotypes      3
Inferiority         Threats      3
      Irony     Inferiority     12
      Irony       Grievance      6
      Irony         Threats      5
      Irony      Incitement      4
      Irony     Stereotypes      4
Stereotypes       Grievance     15
Stereotypes      Incitement     11
Stereotypes     Inferiority      6
Stereotypes           Irony      3
Stereotypes       

In [ ]:
import pandas as pd
import os


project_dir = "/content/drive/MyDrive/implicit_hate_project"
input_file = os.path.join(project_dir, "misclassified_examples.csv")
output_file = os.path.join(project_dir, "major_confusion.csv")


major_pairs = [
    ("Grievance", "Incitement", 22),
    ("Grievance", "Stereotypes", 14),
    ("Grievance", "Inferiority", 8),
    ("Grievance", "Threats", 8),
    ("Incitement", "Grievance", 17),
    ("Incitement", "Stereotypes", 15),
    ("Incitement", "Threats", 12),
    ("Inferiority", "Irony", 14),
    ("Inferiority", "Incitement", 9),
    ("Inferiority", "Grievance", 5),
    ("Irony", "Inferiority", 12),
    ("Irony", "Grievance", 6),
    ("Irony", "Threats", 5),
    ("Stereotypes", "Grievance", 15),
    ("Stereotypes", "Incitement", 11),
    ("Stereotypes", "Inferiority", 6),
    ("Threats", "Incitement", 7)
]


df = pd.read_csv(input_file)

print("Loaded:", len(df), "misclassified examples")


selected_parts = []

for true_label, predicted_label, expected_count in major_pairs:

    subset = df[
        (df["true_label"] == true_label) &
        (df["predicted_label"] == predicted_label)
    ].copy()

    actual_count = len(subset)

    print(
        f"{true_label} → {predicted_label}: "
        f"{actual_count} examples (expected {expected_count})"
    )


    subset.insert(
        0,
        "confusion_pair",
        f"{true_label} → {predicted_label}"
    )

    selected_parts.append(subset)


major_confusion = pd.concat(selected_parts, ignore_index=True)


major_confusion["annotation"] = ""
major_confusion["notes"] = ""


major_confusion = major_confusion[
    [
        "confusion_pair",
        "true_label",
        "predicted_label",
        "post",
        "annotation",
        "notes"
    ]
]


major_confusion.to_csv(output_file, index=False)

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)
print("Saved to:")
print(output_file)
print("\nTotal examples:", len(major_confusion))
print("Expected examples: 177")



Loaded: 221 misclassified examples
Grievance → Incitement: 22 examples (expected 22)
Grievance → Stereotypes: 14 examples (expected 14)
Grievance → Inferiority: 8 examples (expected 8)
Grievance → Threats: 8 examples (expected 8)
Incitement → Grievance: 17 examples (expected 17)
Incitement → Stereotypes: 15 examples (expected 15)
Incitement → Threats: 12 examples (expected 12)
Inferiority → Irony: 14 examples (expected 14)
Inferiority → Incitement: 9 examples (expected 9)
Inferiority → Grievance: 5 examples (expected 5)
Irony → Inferiority: 12 examples (expected 12)
Irony → Grievance: 6 examples (expected 6)
Irony → Threats: 5 examples (expected 5)
Stereotypes → Grievance: 15 examples (expected 15)
Stereotypes → Incitement: 11 examples (expected 11)
Stereotypes → Inferiority: 6 examples (expected 6)
Threats → Incitement: 7 examples (expected 7)

DONE
Saved to:
/content/drive/MyDrive/implicit_hate_project/major_confusion.csv

Total examples: 186
Expected examples: 177


In [ ]:
from google.colab import files

file_path = "/content/drive/MyDrive/implicit_hate_project/major_confusion.csv"

files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>